# Диффузионные методы переноса стиля — этап 1

Один последовательный запуск: данные → smoke test → 24 генерации → метрики → PDF. Основная логика находится в Python-модулях репозитория. LoRA и динамические коэффициенты здесь не реализованы.

Перед запуском сохраните код проекта в указанном GitHub-репозитории. Выберите **Среда выполнения → Сменить среду выполнения → T4 GPU**. Повторный запуск использует сохранённые данные и результаты в Drive.


## 1. Проверка окружения

Проверяем Python и наличие GPU. Модели на этом шаге не загружаются.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

assert (3, 11) <= sys.version_info[:2] <= (3, 13), "Нужен Python 3.11–3.13 для закреплённого окружения"
subprocess.run(["nvidia-smi"], check=True)
print(sys.version)


## 2. Google Drive и расположение артефактов

Данные, кэш моделей, результаты и отчёт сохраняются между перезапусками. Окружение Python и checkout кода находятся на локальном диске Colab.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
HOME_DIR = Path("/content/drive/MyDrive/diffusion_style_transfer")
HOME_DIR.mkdir(parents=True, exist_ok=True)
os.environ["DIFFUSION_STYLE_HOME"] = str(HOME_DIR)
os.environ["HF_HOME"] = str(HOME_DIR / ".cache/huggingface")
os.environ["MPLBACKEND"] = "Agg"
print("Артефакты:", HOME_DIR)


## 3. Загрузка или обновление репозитория

По умолчанию используется main. Для повторения конкретного состояния задайте GIT_REF полным commit SHA. Обновление допускается только для чистого checkout; пользовательские изменения автоматически не стираются.


In [ ]:
REPO_URL = "https://github.com/Batyaoftheyear/diffusion-style.git"
GIT_REF = "main"
REPO = Path("/content/diffusion-style")


def git_command(*arguments, cwd=None):
    result = subprocess.run(
        ["git", *arguments], cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )
    if result.returncode:
        raise RuntimeError(
            f"Git завершился с кодом {result.returncode}: git {' '.join(arguments)}\n"
            f"{result.stdout.strip()}"
        )
    return result.stdout.strip()


remote_refs = git_command("ls-remote", REPO_URL)
if not remote_refs:
    raise RuntimeError(
        "GitHub-репозиторий пустой: удалённых веток и коммитов ещё нет. "
        "Сначала опубликуйте подготовленные файлы проекта в ветке main, "
        "затем повторите эту ячейку. Локальные файлы не появляются на GitHub автоматически."
    )

if not REPO.exists():
    print(git_command("clone", REPO_URL, str(REPO)))
assert (REPO / ".git").is_dir(), "Каталог должен быть отдельным git checkout"
remote = git_command("remote", "get-url", "origin", cwd=REPO)
assert remote.removesuffix(".git") == REPO_URL.removesuffix(".git"), "Другой origin"
status = git_command("status", "--porcelain", cwd=REPO)
assert not status, "В checkout есть изменения; сохраните их перед обновлением"
print(git_command("fetch", "origin", GIT_REF, cwd=REPO))
print(git_command("checkout", "--detach", "FETCH_HEAD", cwd=REPO))
print(git_command("rev-parse", "HEAD", cwd=REPO))


## 4. Установка закреплённых зависимостей

Используется отдельное окружение с pip, созданное автономным virtualenv: системный ensurepip не требуется. Проверяется контрольная сумма bootstrap-архива; частично созданное окружение автоматически дополняется pip. Предустановленные библиотеки Colab не заменяются. Вывод установки и ошибки видны в ячейке. Скачивание CUDA-библиотек и весов требует свободного места и времени.


In [ ]:
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.bootstrap import ensure_environment, requirements_fingerprint, run_command

VENV = Path("/content/diffusion-style-env")
PYTHON = ensure_environment(VENV)
requirements = REPO / "requirements.txt"
requirements_sha = requirements_fingerprint(requirements)
marker = VENV / "requirements.sha256"
if not marker.exists() or marker.read_text().strip() != requirements_sha:
    run_command([PYTHON, "-m", "pip", "install", "-r", requirements], cwd=REPO)
    run_command([PYTHON, "-m", "pip", "check"], cwd=REPO)
    marker.write_text(requirements_sha)
else:
    run_command([PYTHON, "-m", "pip", "check"], cwd=REPO)
run_command([PYTHON, "-c", "import torch, diffusers, transformers; print(torch.__version__, diffusers.__version__, transformers.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"], cwd=REPO)


def run_script(name, *arguments):
    run_command([PYTHON, REPO / "scripts" / name, *arguments], cwd=REPO)


## 5. CPU-тесты до загрузки моделей

Тесты проверяют preprocessing, выборку, метрики, manifests, latents, resume и черновик PDF. GPU smoke test по умолчанию пропускается; он выполняется далее настоящим запуском пайплайна.


In [ ]:
run_command([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO)


## 6. Подготовка и проверка данных

Загружаются 75 картин импрессионизма и 24 фото COCO, а не полные наборы изображений. Пять картин выделяются в референсы. Повторный запуск проверяет hashes и использует подготовленные файлы.


In [ ]:
run_script("prepare_data.py")
run_script("prepare_data.py", "--check-only")


## 7. Просмотр данных

Проверьте центральную обрезку, соответствие captions фотографиям и однородность картин до запуска baseline. Главный референс выбран детерминированно; изменения исследования фиксируйте в новой конфигурации.


In [ ]:
import json
from IPython.display import display, Image as DisplayImage

DATA_DIR = Path(subprocess.check_output([str(PYTHON), "-c", "from src.config import load_config,data_dir; print(data_dir(load_config()))"], cwd=REPO, text=True).strip())
display(DisplayImage(filename=str(DATA_DIR / "preview.png"), width=768))
art = [json.loads(line) for line in (DATA_DIR / "art_manifest.jsonl").read_text().splitlines()]
photos = [json.loads(line) for line in (DATA_DIR / "coco_manifest.jsonl").read_text().splitlines()]
print("Картины:", len(art), "Референсы:", sum(row["split"] == "reference" for row in art), "Фото:", len(photos))
for row in photos[:6]:
    print(row["id"], row["scene_group"], "—", row["prompt"])


## 8. Загрузка моделей и настоящий GPU smoke test

Скрипт загружает закреплённые SD 1.5, ControlNet и IP-Adapter, формирует фиксированный шум и генерирует две фотографии. Если этап завершился ошибкой, исправьте причину до полного запуска. Существующие успешные результаты проверяются и переиспользуются.


In [ ]:
run_script("run_baseline.py", "--limit", "2")
RUN_DIR = HOME_DIR / "results" / json.loads((HOME_DIR / "results/current_run.json").read_text())["run_id"]
display(DisplayImage(filename=str(RUN_DIR / "figures/comparisons.png"), width=1000))


## 9. Основной baseline на 24 фото

Выполняются оставшиеся генерации с той же конфигурацией. После перезапуска Colab снова выполните ячейки сверху вниз: повторные скачивания данных и готовые генерации будут пропущены после проверки. Не запускайте несколько runtimes с записью в один каталог.

Если последний результат остановил safety checker, уже сохранённые изображения сохраняются. Для подтверждённой блокировки можно выполнить отдельной командой run_script("finalize_run.py", "--skip-photo", "coco_000000423971", "--reason", "safety_checker"). Она требует готовых результатов для всех остальных фото, отмечает пропуск в текущем run_id и позволяет перейти к метрикам по фактическому n. Повторная генерация не нужна; в отчёте пропуск будет указан явно.


In [ ]:
run_script("run_baseline.py")
RUN_DIR = HOME_DIR / "results" / json.loads((HOME_DIR / "results/current_run.json").read_text())["run_id"]
print(json.loads((RUN_DIR / "run.json").read_text())["completed_images"], "из 24")


## 10. Метрики и визуализация

Генерационные модели уже выгружены при завершении предыдущего процесса. DINOv2 запускается отдельно. Разброс в summary относится к фотографиям при одном seed, а не к устойчивости к шуму.


In [ ]:
run_script("evaluate.py")
summary = json.loads((RUN_DIR / "summary.json").read_text())
for name, values in summary["metrics"].items():
    print(f"{name}: {values['mean']:.4f} ± {values['std']:.4f}; n={values['n']}")
display(DisplayImage(filename=str(RUN_DIR / "figures/comparisons.png"), width=1000))
print("CSV:", RUN_DIR / "metrics.csv")


## 11. Итоговый PDF

Обычный режим требует 24 действительных генерации и соответствующие метрики. Проверяются объём 4–6 страниц и извлечение кириллицы, затем создаются PNG страниц. Перед сдачей просмотрите их и сам PDF. Если GPU ещё не запускался, локально допускается только команда build_report.py --draft; такой документ не готов к сдаче.


In [ ]:
run_script("build_report.py", "--render")
PDF_PATH = HOME_DIR / "reports/stage1_report.pdf"
for preview in sorted((HOME_DIR / "reports/preview").glob("page_*.png")):
    display(DisplayImage(filename=str(preview), width=700))
print("PDF для ЛМС:", PDF_PATH)
print("Метаданные PDF:", json.loads(PDF_PATH.with_suffix(".json").read_text()))


## 12. Скачать отчёт

Результаты также остаются в MyDrive/diffusion_style_transfer. В ЛМС передаются окончательный PDF и ссылка на GitHub-репозиторий.


In [ ]:
from google.colab import files
files.download(str(PDF_PATH))
